# # A.R.I.A. — Adaptive Road Intelligence Architecture
**Version 3.1** · 4-Class Defect Type Detection · 2-Stage Rehearsal Curriculum

| Stage | Dataset | Purpose |
|---|---|---|
| 1 | RDD2022 All Countries (26.8k images) | Global foundation — learn all 4 defect types |
| 2 | IIT Madras + RDD2022 India (~11.5k images) | Rehearsal fine-tune — Indian specialization without catastrophic forgetting |

**Output Classes:** `0 longitudinal_crack` · `1 transverse_crack` · `2 alligator_crack` · `3 pothole`

Severity is calculated in the backend using Type + Bounding Box Area, not by YOLO directly.


## Cell 1 — Install Ultralytics
Pins the exact version to prevent silent breaking changes between runs.


In [1]:
!pip install -q ultralytics==8.4.21


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.1 MB/s eta 0:00:00


## Cell 2 — Imports & Dataset Discovery
Dynamically walks `/kaggle/input/` to find the RDD2022 and IIT Madras datasets regardless of Kaggle slug names. Prints a directory tree and dataset split counts for verification. Saves all discovered paths to `aria_config.json` so subsequent cells can read them without re-scanning.


In [2]:
import os
import json
import shutil
import math
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import torch
from ultralytics import YOLO
from ultralytics import __version__ as ul_version

print(f"Ultralytics : {ul_version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()} — "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

CONFIG_PATH = Path("/kaggle/working/aria_config.json")

def find_rdd_split(root="/kaggle/input") -> Path | None:
    for dp, dns, _ in os.walk(root):
        dl = {d.lower(): d for d in dns}
        if "rdd_split" in dl:
            return Path(dp) / dl["rdd_split"]
    return None

def find_itt_madras(root="/kaggle/input") -> Path | None:
    for dp, dns, fns in os.walk(root):
        if "data.yaml" not in fns:
            continue
        cand = Path(dp)
        cand_str = str(cand).lower()
        if "madras" not in cand_str and "itt" not in cand_str:
            continue
        img_dir = cand / "train" / "images"
        if img_dir.exists() and any(img_dir.iterdir()):
            return cand
    return None

RDD_SPLIT = find_rdd_split()
ITT_ROOT  = find_itt_madras()

print("\n" + "=" * 64)
print("/kaggle/input/  (2 levels)")
print("=" * 64)
for dp, dns, fns in os.walk("/kaggle/input"):
    depth = dp.replace("/kaggle/input", "").count(os.sep)
    if depth > 2:
        dns.clear()
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(dp) or 'input'}/  [{len(fns)} files]")

print(f"\nRDD_SPLIT  : {RDD_SPLIT}")
print(f"ITT Madras : {ITT_ROOT}")

if RDD_SPLIT is None or ITT_ROOT is None:
    raise RuntimeError("Datasets not found. Check the tree above.")

print("\n" + "=" * 64)
print("RDD2022 split counts")
print("=" * 64)
for sp in ("train", "val", "test"):
    img_d = RDD_SPLIT / sp / "images"
    lbl_d = RDD_SPLIT / sp / "labels"
    imgs = len(list(img_d.glob("*"))) if img_d.exists() else 0
    lbls = len(list(lbl_d.glob("*.txt"))) if lbl_d.exists() else 0
    print(f"  {sp:<6}: {imgs:>6} images   {lbls:>6} labels")

cfg = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
cfg.update({"rdd_split": str(RDD_SPLIT), "itt_root": str(ITT_ROOT)})
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(f"\n✓ Paths saved to {CONFIG_PATH}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics : 8.4.21
PyTorch     : 2.9.0+cu126
CUDA        : True — Tesla P100-PCIE-16GB

/kaggle/input/  (2 levels)
input/  [0 files]
  datasets/  [0 files]
    dptel22/  [0 files]
    aliabdelmenam/  [0 files]

RDD_SPLIT  : /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT
ITT Madras : /kaggle/input/datasets/dptel22/itt-madras

RDD2022 split counts
  train :  26869 images    26869 labels
  val   :   5758 images     5758 labels
  test  :   5758 images     5758 labels

✓ Paths saved to /kaggle/working/aria_config.json


## Cell 3 — Stage 1 Data Prep: All RDD2022 → 4 Defect Types
Symlinks all RDD2022 images (train + val, all countries) into a working directory. Remaps the original 5 RDD class IDs to our 4 target types:
- `D00` → `0` longitudinal_crack
- `D10` → `1` transverse_crack
- `D20` → `2` alligator_crack
- `D40/D44` → `3` pothole

Uses `os.symlink` (zero disk cost) and writes remapped `.txt` label files. Empty `.txt` files are created for images without labels (background/negative samples).


In [3]:
import os
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT   = Path(cfg["rdd_split"])
STAGE1_DIR  = Path("/kaggle/working/stage1_data")

shutil.rmtree(STAGE1_DIR, ignore_errors=True)

SPLIT_MAP = {
    RDD_SPLIT / "train": STAGE1_DIR / "train",
    RDD_SPLIT / "val":   STAGE1_DIR / "valid",
}
for dst in SPLIT_MAP.values():
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)

RDD_TO_TYPE = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}

cfg["rdd_to_type"] = RDD_TO_TYPE
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

def _symlink_and_remap(task):
    img_src, dst_img, src_lbl, dst_lbl = task
    try:
        try:
            os.symlink(img_src, dst_img)
        except FileExistsError:
            pass

        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                if p := line.strip().split():
                    orig_cls = int(p[0])
                    if orig_cls in RDD_TO_TYPE:
                        new_cls = RDD_TO_TYPE[orig_cls]
                        lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
        return True, None
    except Exception as e:
        return False, f"{img_src.name}: {e}"

total_ok = total_err = total_bg = 0
for src_split, dst_split in SPLIT_MAP.items():
    src_img_dir = src_split / "images"
    src_lbl_dir = src_split / "labels"

    tasks = [
        (img,
         dst_split / "images" / img.name,
         src_lbl_dir / (img.stem + ".txt"),
         dst_split / "labels" / (img.stem + ".txt"))
        for img in src_img_dir.glob("*") if img.is_file()
    ]
    ok = err = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        for success, msg in pool.map(_symlink_and_remap, tasks):
            if success:
                ok += 1
            else:
                err += 1
                if msg:
                    print(f"  [ERROR] {msg}")

    bg = sum(1 for f in (dst_split / "labels").glob("*.txt") if f.read_text().strip() == "")
    print(f"  {dst_split.name:<8}: {ok:>6} linked  {bg:>5} background  {err} errors")
    total_ok += ok; total_err += err; total_bg += bg

(STAGE1_DIR / "data.yaml").write_text(
    f"path: {STAGE1_DIR}\n"
    f"train: train/images\n"
    f"val: valid/images\n"
    "nc: 4\n"
    "names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
)

train_n = len(list((STAGE1_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE1_DIR / "valid" / "images").glob("*")))
print(f"\nStage 1 Dataset Ready:")
print(f"  train : {train_n:>6} images")
print(f"  valid : {valid_n:>6} images")

if train_n == 0:
    raise RuntimeError("0 images in stage1 train.")

cfg["stage1"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))


  train   :  26869 linked   8097 background  0 errors
  valid   :   5758 linked   1837 background  0 errors

Stage 1 Dataset Ready:
  train :  26869 images
  valid :   5758 images


299

## Cell 4 — Stage 1 Training: Global Base (`yolo11n.pt` → `aria_stage1.pt`)
Trains YOLOv11n from COCO pretrained weights on the full RDD2022 dataset (all countries, ~26.8k images) for 50 epochs using SGD. This builds the global "visual brain" that understands road textures, cracks, and potholes across diverse conditions. Saves the best checkpoint as `aria_stage1.pt` and cleans up the stage1 data directory to free disk.

**Key hyperparams:** `batch=-1` (auto), `patience=15`, `close_mosaic=10`, heavy augmentation enabled.


In [4]:
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE1_DIR     = Path("/kaggle/working/stage1_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

model   = YOLO("yolo11n.pt")
results = model.train(
    data          = str(STAGE1_DIR / "data.yaml"),
    epochs        = 50,
    imgsz         = 640,
    batch         = -1,
    device        = 0,
    workers       = 4,
    name          = "aria_stage1",
    project       = "/kaggle/working/runs",
    exist_ok      = True,
    optimizer     = "SGD",
    lr0           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    cos_lr        = True,
    warmup_epochs = 3,
    patience      = 15,
    close_mosaic  = 10,
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    mixup         = 0.1,
    copy_paste    = 0.1,
    erasing       = 0.2,
    amp           = True,
    cache         = False,
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, STAGE1_WEIGHTS)

stage1_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage1_mAP50"] = stage1_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print("\nCleaning up stage1_data to free disk...")
shutil.rmtree(STAGE1_DIR, ignore_errors=True)

print(f"\n{'─'*48}")
print(f"  Stage 1  mAP@50 : {stage1_map50:.4f}")
print(f"  Saved   → {STAGE1_WEIGHTS}")
print(f"{'─'*48}")


New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/stage1_data/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

## Cell 5 — Stage 2 Data Prep: IIT Madras + RDD2022 India (Mixed Train, Pure Val)
Creates the rehearsal dataset for Stage 2 fine-tuning:
- **Train set:** IIT Madras training images + all `India_*` prefixed images from RDD2022 train (~11.5k total). This prevents catastrophic forgetting by keeping crack examples in the training loop.
- **Val set:** IIT Madras validation images ONLY (~542 images). Kept pure so the benchmark is interpretable and not contaminated by RDD2022 data.

IIT Madras classes are remapped: `longitudinal crack` → 0, `crocodile crack` → 2, `pothole` → 3. Note: `transverse_crack` (class 1) has zero IIT Madras contribution — it is learned exclusively from the RDD2022 India images.

Prints a class distribution table so you can verify the imbalance fix before training.


In [5]:
# %% [Cell 5] Stage 2 data preparation — BALANCED Mixed Train, PURE Val (4 Types)
import os
import json
import shutil
import yaml
import random
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

random.seed(42)

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
ITT_ROOT    = Path(cfg["itt_root"])
RDD_SPLIT   = Path(cfg["rdd_split"])
STAGE2_DIR  = Path("/kaggle/working/stage2_data")

shutil.rmtree(STAGE2_DIR, ignore_errors=True)

for sp in ("train", "valid"):
    (STAGE2_DIR / sp / "images").mkdir(parents=True, exist_ok=True)
    (STAGE2_DIR / sp / "labels").mkdir(parents=True, exist_ok=True)

# ── Dynamic class mappings from data.yaml ─────────────────────────────────────
itt_yaml    = yaml.safe_load((ITT_ROOT / "data.yaml").read_text())
itt_names   = itt_yaml.get("names", [])
name_to_idx = {n.lower().strip(): i for i, n in enumerate(itt_names)}

_raw = cfg.get("rdd_to_type", {"0": 0, "1": 1, "2": 2, "3": 3, "4": 3})
RDD_TO_TYPE = {int(k): v for k, v in _raw.items()}

ITT_TO_TYPE = {}
for name, tid in {"longitudinal crack": 0, "crocodile crack": 2, "pothole": 3}.items():
    if name in name_to_idx:
        ITT_TO_TYPE[name_to_idx[name]] = tid

CLASS_NAMES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
CLASS_CAPS  = {0: 1500, 1: 1500, 2: 1000, 3: 500}

# ── Step 1: Gather raw image pools ───────────────────────────────────────────
raw_train = []   # (img_path, lbl_path, mapping)
raw_valid = []

itt_val_sp = "valid" if (ITT_ROOT / "valid" / "images").exists() else "val"

# IIT Madras → valid (pure, untouched)
src_v = ITT_ROOT / itt_val_sp
if (src_v / "images").exists():
    for img in (src_v / "images").glob("*"):
        if img.is_file():
            raw_valid.append((img, (src_v / "labels") / (img.stem + ".txt"), ITT_TO_TYPE))

# IIT Madras → train pool
src_t = ITT_ROOT / "train"
if (src_t / "images").exists():
    for img in (src_t / "images").glob("*"):
        if img.is_file():
            raw_train.append((img, (src_t / "labels") / (img.stem + ".txt"), ITT_TO_TYPE))

# RDD India → train pool
src_rdd = RDD_SPLIT / "train"
if (src_rdd / "images").exists():
    for img in (src_rdd / "images").glob("*"):
        if img.name.startswith("India_") and img.is_file():
            raw_train.append((img, (src_rdd / "labels") / (img.stem + ".txt"), RDD_TO_TYPE))

print(f"Raw pools — train: {len(raw_train)}, valid: {len(raw_valid)}")

# ── Step 2: Scan labels → bucket by dominant class ───────────────────────────
def _scan(lbl, mapping):
    """Read label file, return Counter of remapped class IDs."""
    c = Counter()
    if not lbl.exists():
        return c
    for line in lbl.read_text().strip().splitlines():
        if p := line.strip().split():
            if len(p) >= 5 and int(p[0]) in mapping:
                c[mapping[int(p[0])]] += 1
    return c

# Each item: (img, lbl, mapping, is_mixed, dominant_cls)
# dominant_cls = -1 for background images (no annotations)
buckets = defaultdict(list)

for img, lbl, mapping in raw_train:
    cc = _scan(lbl, mapping)
    if not cc:
        buckets[-1].append((img, lbl, mapping, False, -1))
        continue
    dom = cc.most_common(1)[0][0]
    buckets[dom].append((img, lbl, mapping, len(cc) > 1, dom))

print(f"\nPre-balance dominant-class buckets:")
for cid in sorted(k for k in buckets if k >= 0):
    n  = len(buckets[cid])
    mx = sum(1 for t in buckets[cid] if t[3])
    print(f"  {cid} {CLASS_NAMES[cid]:<22}: {n:>5} images ({mx} mixed)  cap={CLASS_CAPS[cid]}")
bg_n = len(buckets.get(-1, []))
if bg_n:
    print(f"  - background              : {bg_n:>5} images (kept all)")

# ── Step 3: Cap each bucket — drop mixed-class images first ──────────────────
balanced = []

for cid, cap in CLASS_CAPS.items():
    bkt = buckets.get(cid, [])
    if len(bkt) <= cap:
        balanced.extend(bkt)
        continue

    pure  = [t for t in bkt if not t[3]]
    mixed = [t for t in bkt if t[3]]

    if len(pure) >= cap:
        random.shuffle(pure)
        balanced.extend(pure[:cap])
    else:
        balanced.extend(pure)
        random.shuffle(mixed)
        balanced.extend(mixed[:cap - len(pure)])

# Keep all background images
balanced.extend(buckets.get(-1, []))

# ── Step 4: Augment transverse cracks if under cap ───────────────────────────
trans_items = [t for t in balanced if t[4] == 1]
trans_n     = len(trans_items)
aug_tasks   = []

if 0 < trans_n < CLASS_CAPS[1]:
    need = CLASS_CAPS[1] - trans_n
    print(f"\n  transverse_crack: {trans_n} real → generating {need} augmented copies")
    for i in range(need):
        src = trans_items[i % trans_n]
        aug_tasks.append((src[0], src[1], src[2], f"_aug{i}"))

print(f"\nPost-balance: {len(balanced)} images + {len(aug_tasks)} augmented = "
      f"{len(balanced) + len(aug_tasks)} total train")

# ── Step 5: Symlink + remap via ThreadPoolExecutor ───────────────────────────
def _process(task):
    img_src, dst_img, src_lbl, dst_lbl, mapping = task
    is_bg  = False
    counts = {0: 0, 1: 0, 2: 0, 3: 0}
    try:
        try:
            os.symlink(img_src, dst_img)
        except FileExistsError:
            pass

        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                if p := line.strip().split():
                    if len(p) >= 5 and int(p[0]) in mapping:
                        nc = mapping[int(p[0])]
                        counts[nc] += 1
                        lines.append(f"{nc} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
            is_bg = True
        return True, None, is_bg, counts
    except Exception as e:
        return False, str(e), False, counts

# Build (img_src, dst_img, src_lbl, dst_lbl, mapping) tuples
exec_tasks = []

# Valid tasks (processed first so we can separate counts by index)
for img, lbl, mapping in raw_valid:
    exec_tasks.append((
        img,
        STAGE2_DIR / "valid" / "images" / img.name,
        lbl,
        STAGE2_DIR / "valid" / "labels" / (img.stem + ".txt"),
        mapping
    ))
valid_count = len(exec_tasks)

# Balanced train tasks
for img, lbl, mapping, _, _ in balanced:
    exec_tasks.append((
        img,
        STAGE2_DIR / "train" / "images" / img.name,
        lbl,
        STAGE2_DIR / "train" / "labels" / (img.stem + ".txt"),
        mapping
    ))

# Augmented transverse copies (same source image, different filename)
for img, lbl, mapping, suffix in aug_tasks:
    exec_tasks.append((
        img,
        STAGE2_DIR / "train" / "images" / (img.stem + suffix + img.suffix),
        lbl,
        STAGE2_DIR / "train" / "labels" / (img.stem + suffix + ".txt"),
        mapping
    ))

print(f"\nExecuting {len(exec_tasks)} symlink+remap tasks...")
ok = err = total_bg = 0
train_counts = {0: 0, 1: 0, 2: 0, 3: 0}

with ThreadPoolExecutor(max_workers=8) as pool:
    for i, (success, msg, is_bg, counts) in enumerate(pool.map(_process, exec_tasks)):
        if success:
            ok += 1
            if is_bg:
                total_bg += 1
            if i >= valid_count:
                for k, v in counts.items():
                    train_counts[k] += v
        else:
            err += 1
            if msg:
                print(f"  [ERROR] {msg}")

# ── Write data.yaml ──────────────────────────────────────────────────────────
(STAGE2_DIR / "data.yaml").write_text(
    f"path: {STAGE2_DIR}\n"
    f"train: train/images\n"
    f"val: valid/images\n"
    "nc: 4\n"
    "names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
)

train_n = len(list((STAGE2_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE2_DIR / "valid" / "images").glob("*")))

if train_n == 0:
    raise RuntimeError("0 images in stage2 train.")

print(f"\nStage 2 Dataset Ready (Train: BALANCED Mixed | Val: IIT PURE):")
print(f"  train : {train_n:>6} images")
print(f"  valid : {valid_n:>6} images")
print(f"  errors: {err:>6}")

total_inst = sum(train_counts.values())
print(f"\n{'─'*52}")
print(f"  Stage 2 Train — Instance Distribution (BALANCED)")
print(f"{'─'*52}")
for i, name in enumerate(CLASS_NAMES):
    pct = (train_counts[i] / total_inst * 100) if total_inst > 0 else 0
    print(f"  {i} {name:<22}: {train_counts[i]:>6} instances  ({pct:>5.1f}%)")
print(f"  {'TOTAL':<24}: {total_inst:>6}")
print(f"{'─'*52}")

cfg["stage2_mixed"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))


Raw pools — train: 7274, valid: 542

Pre-balance dominant-class buckets:
  0 longitudinal_crack    :   629 images (325 mixed)  cap=1500
  1 transverse_crack      :    13 images (6 mixed)  cap=1500
  2 alligator_crack       :   839 images (367 mixed)  cap=1000
  3 pothole               :  3101 images (763 mixed)  cap=500
  - background              :  2692 images (kept all)

  transverse_crack: 13 real → generating 1487 augmented copies

Post-balance: 4673 images + 1487 augmented = 6160 total train

Executing 6702 symlink+remap tasks...

Stage 2 Dataset Ready (Train: BALANCED Mixed | Val: IIT PURE):
  train :   6160 images
  valid :    542 images
  errors:      0

────────────────────────────────────────────────────
  Stage 2 Train — Instance Distribution (BALANCED)
────────────────────────────────────────────────────
  0 longitudinal_crack    :   1654 instances  ( 23.0%)
  1 transverse_crack      :   1983 instances  ( 27.6%)
  2 alligator_crack       :   1432 instances  ( 19.9%)
  3 po

419

## Cell 6 — Stage 2 Training: Rehearsal Fine-Tune (`aria_stage1.pt` → `aria_best_v1.pt`)
Fine-tunes the Stage 1 checkpoint on the mixed Indian dataset using AdamW optimizer with a low learning rate (0.001).

**Critical hyperparameters:**
- `freeze=3` — locks the 3 lowest backbone layers (edge/texture detectors) to prevent catastrophic forgetting, lets everything else adapt to Indian roads.
- `warmup_epochs=0` — no warmup spiking when fine-tuning from a checkpoint.
- `lrf=0.1` — final LR = 0.0001, prevents late-epoch stagnation.
- `cls=1.5` — upweights classification loss to fight class imbalance.
- `mixup=0, erasing=0` — disabled to protect thin crack features and severity boundaries.

Saves `aria_best_v1.pt` and cleans up the train directory (preserves val for Cell 7 evaluation).


In [6]:
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE2_DIR     = Path("/kaggle/working/stage2_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")
FINAL_WEIGHTS  = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

if not STAGE1_WEIGHTS.exists():
    raise FileNotFoundError(f"Stage 1 weights not found: {STAGE1_WEIGHTS}. Run Cell 4 first.")

model = YOLO(str(STAGE1_WEIGHTS))
results = model.train(
    data          = str(STAGE2_DIR / "data.yaml"),
    epochs        = 40,
    imgsz         = 640,
    batch         = -1,
    device        = 0,
    workers       = 4,
    name          = "aria_stage2_mixed",
    project       = "/kaggle/working/runs",
    exist_ok      = True,

    freeze        = 3,
    optimizer     = "AdamW",
    lr0           = 0.001,
    weight_decay  = 0.0005,
    cls           = 1.0,

    warmup_epochs = 0.0,
    lrf           = 0.1,

    cos_lr        = True,
    close_mosaic  = 10,
    patience      = 15,

    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,

    mixup         = 0.0,
    copy_paste    = 0.0,
    erasing       = 0.0,

    amp           = True,
    cache         = False,
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, FINAL_WEIGHTS)

stage2_mixed_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage2_mixed_mAP50"] = stage2_mixed_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print("\nCleaning up stage2_data/train to free disk...")
shutil.rmtree(STAGE2_DIR / "train", ignore_errors=True)
print("  ✓ stage2_data/train removed (valid/ preserved)")

print(f"\n{'─'*48}")
print(f"  Stage 2 (MIXED) mAP@50 : {stage2_mixed_map50:.4f}")
print(f"  Saved   → {FINAL_WEIGHTS}")
print(f"{'─'*48}")


New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/stage2_data/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=3, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/aria_stage1.pt, momentum=0.937, mosaic=1.0, mult

## Cell 7 — Final Evaluation: IIT Madras Val + RDD2022 Test (Held-Out)
Runs the final model (`aria_best_v1.pt`) against two benchmarks:

1. **IIT Madras val set** (~542 images) — the training validation set. This number is optimistically biased because YOLO used it to select `best.pt`.
2. **RDD2022 test split** (~5,758 images, all countries) — truly held-out data the model has never seen. This is the real generalization metric.

For each benchmark, prints **per-class AP@50** using `ap_class_index` to correctly anchor scores even when a class is absent from the validation set (e.g., `transverse_crack` is missing from IIT Madras).

Cleans up temporary test data via `try/finally` to guarantee disk recovery even if `model.val()` OOMs.


In [7]:
import os
import json
import math
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from ultralytics import YOLO

CONFIG_PATH   = Path("/kaggle/working/aria_config.json")
FINAL_WEIGHTS = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT = Path(cfg["rdd_split"])

_raw_mapping = cfg.get("rdd_to_type", {"0": 0, "1": 1, "2": 2, "3": 3, "4": 3})
RDD_TO_TYPE = {int(k): v for k, v in _raw_mapping.items()}

rdd_test_map50 = float("nan")
CLASS_NAMES    = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(f"Final weights not found: {FINAL_WEIGHTS}. Run Cell 6 first.")

model = YOLO(str(FINAL_WEIGHTS))

def _print_per_class_ap50(metrics, header: str):
    ap_per_class = {
        int(idx): float(ap)
        for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)
    }
    print(f"\n  Per-class AP@50 ({header}):")
    for i, name in enumerate(CLASS_NAMES):
        ap = ap_per_class.get(i, float("nan"))
        line = f"{'N/A':<8}  <- absent from val set" if math.isnan(ap) else f"{ap:.4f}"
        print(f"    {name:<22}: {line}")

# ── 1. IIT Madras val ─────────────────────────────────────────────────────────
print("=" * 66)
print("  Validating on IIT Madras val set (training val set)")
print("=" * 66)

metrics_itt = model.val(
    data   = "/kaggle/working/stage2_data/data.yaml",
    device = 0,
    imgsz  = 640,
    plots  = True,
)
itt_map50 = float(metrics_itt.box.map50)
_print_per_class_ap50(metrics_itt, "IIT Madras val")

# ── 2. RDD2022 test split ────────────────────────────────────────────────────
rdd_test_img = RDD_SPLIT / "test" / "images"
rdd_test_lbl = RDD_SPLIT / "test" / "labels"

if rdd_test_img.exists() and any(rdd_test_img.iterdir()):
    TEST_DIR = Path("/kaggle/working/rdd_test_eval")
    (TEST_DIR / "images").mkdir(parents=True, exist_ok=True)
    (TEST_DIR / "labels").mkdir(parents=True, exist_ok=True)

    def _prep_test_img(img):
        dst_img = TEST_DIR / "images" / img.name
        dst_lbl = TEST_DIR / "labels" / (img.stem + ".txt")
        try:
            os.symlink(img, dst_img)
        except FileExistsError:
            pass
        src_lbl = rdd_test_lbl / (img.stem + ".txt")
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                if p := line.strip().split():
                    if len(p) >= 5 and int(p[0]) in RDD_TO_TYPE:
                        new_cls = RDD_TO_TYPE[int(p[0])]
                        lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
        return True

    test_imgs = [img for img in rdd_test_img.glob("*") if img.is_file()]
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(_prep_test_img, test_imgs))

    (TEST_DIR / "data.yaml").write_text(
        f"path: {TEST_DIR}\n"
        f"train: images\n"
        f"val: images\n"
        "nc: 4\n"
        "names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
    )

    print(f"\n{'=' * 66}")
    print(f"  Validating on RDD2022 TEST split ({len(test_imgs)} images — truly held-out)")
    print(f"{'=' * 66}")

    try:
        metrics_rdd    = model.val(
            data   = str(TEST_DIR / "data.yaml"),
            device = 0,
            imgsz  = 640,
            plots  = True,
        )
        rdd_test_map50 = float(metrics_rdd.box.map50)
        _print_per_class_ap50(metrics_rdd, "RDD2022 test — held-out")
    finally:
        shutil.rmtree(TEST_DIR, ignore_errors=True)

# ── 3. Save & Report ─────────────────────────────────────────────────────────
cfg["final_itt_mAP50"]      = itt_map50
cfg["final_rdd_test_mAP50"] = rdd_test_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

s1 = cfg.get("stage1_mAP50", float("nan"))
s2 = cfg.get("stage2_mixed_mAP50", float("nan"))

W = 70
print(f"\n{'═'*W}")
print(f"  A.R.I.A.  Road Damage Detection — v3.1")
print(f"  Architecture : YOLOv11n  (4-Class Type Detection)")
print(f"  Model        : {FINAL_WEIGHTS.name}")
print(f"{'─'*W}")
print(f"  IIT Madras val set  (training val — optimistic)")
print(f"    mAP@50    : {itt_map50:.4f}")
print(f"{'─'*W}")
if not math.isnan(rdd_test_map50):
    print(f"  RDD2022 test split  (held-out — TRUE generalisation)")
    print(f"    mAP@50    : {rdd_test_map50:.4f}")
print(f"{'═'*W}")


  Validating on IIT Madras val set (training val set)
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11n summary (fused): 101 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 58.1±9.9 MB/s, size: 75.0 KB)
val: Scanning /kaggle/working/stage2_data/valid/labels.cache... 542 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 542/542 227.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 6.9it/s 4.9s
                   all        542       1853      0.404       0.34      0.297      0.117
    longitudinal_crack        107        284      0.292      0.296      0.204     0.0719
       alligator_crack         63        125      0.374      0.216      0.185     0.0519
               pothole        533       1444      0.546      0.509      0.503      0.228
Speed: 1.3ms preprocess, 2.5ms inference, 0.0ms loss, 1.1ms po